In [ ]:
import pandas as pd
from factor_analyzer import FactorAnalyzer
import matplotlib.pyplot as plt
import numpy as np
from utils.utils import min_max_normalize

# Carregar dados e preparar variáveis

In [ ]:
df_res = pd.read_excel("./outputs/df_res.xlsx")

df_res["mesa_cum"] = df_res["mesa_1_cum"] + df_res["mesa_2_cum"]


df_res["mandatos_cum"] = (
    df_res["mand_dep_estadual_cum"]
    + df_res["mand_sen_cum"]
    + df_res["mand_dep_estadual_cum"]
)


vars = [
    ("tempo_atuacao_percent", r"$TempoAtuacao$"),
    ("relatorias_ln_cum", r"$ln(Relatoria)$"),
    ("pos_lider_cum", r"$PosicaoLider$"),
    ("pos_comiss_pr_cum", r"$PresidenciaComissao$"),
    ("mesa_cum", r"$Mesa$"),
    ("mandatos_cum", r"$Mandatos$"),
    ("fid_gerais_cum", r"$FidelidadeGerais$"),
]

cols = [v[0] for v in vars]

df_res = df_res.fillna(0)

df_res[cols].describe()

df_final = df_res.iloc[:, 0:8].join(df_res[cols])

# Normalizando e aplicando os pesos

In [ ]:
# normalizar
for col, label in vars:
    df_final[f"{col}_norm"] = min_max_normalize(df_final[col])

# appply weights
weights = {
    "tempo_atuacao_percent_norm": 1,
    "relatorias_ln_cum_norm": 2,
    "pos_lider_cum_norm": 4,
    "pos_comiss_pr_cum_norm": 3,
    "mesa_cum_norm": 5,
    "mandatos_cum_norm": 1,
    "fid_gerais_cum_norm": 1,
}
for var_name, weight in weights.items():
    df_final[f"{var_name}_w"] = df_final[var_name] * weight

df_final.iloc[:, 8:].describe().T.sort_index()

,count,mean,std,min,25%,50%,75%,max
fid_gerais_cum,6250.0,1.297280,1.340057,0.0,0.000000,1.000000,2.000000,7.000000
fid_gerais_cum_norm,6250.0,0.185326,0.191437,0.0,0.000000,0.142857,0.285714,1.000000
fid_gerais_cum_norm_w,6250.0,0.185326,0.191437,0.0,0.000000,0.142857,0.285714,1.000000
mandatos_cum,6250.0,1.248800,2.139633,0.0,0.000000,0.000000,2.000000,14.000000
mandatos_cum_norm,6250.0,0.089200,0.152831,0.0,0.000000,0.000000,0.142857,1.000000
mandatos_cum_norm_w,6250.0,0.089200,0.152831,0.0,0.000000,0.000000,0.142857,1.000000
mesa_cum,6250.0,0.026560,0.210004,0.0,0.000000,0.000000,0.000000,5.000000
mesa_cum_norm,6250.0,0.005312,0.042001,0.0,0.000000,0.000000,0.000000,1.000000
mesa_cum_norm_w,6250.0,0.026560,0.210004,0.0,0.000000,0.000000,0.000000,5.000000
pos_comiss_pr_cum,6250.0,0.254560,0.600314,0.0,0.000000,0.000000,0.000000,5.000000


# Calculando dimensões e índice final

In [45]:
dimensoes = {
    "dim_comprometimento": [
        "mesa_cum_norm_w",
        "pos_lider_cum_norm_w",
        "pos_comiss_pr_cum_norm_w",
        "relatorias_ln_cum_norm",
    ],
    "dim_carreira": [
        "mandatos_cum_norm_w",
        "fid_gerais_cum_norm_w",
        "tempo_atuacao_percent_norm_w",
    ],
}

df_final["dim_comprometimento"] = (
    df_final["mesa_cum_norm_w"]
    + df_final["pos_lider_cum_norm_w"]
    + df_final["pos_comiss_pr_cum_norm_w"]
    + df_final["relatorias_ln_cum_norm"]
)

df_final["dim_carreira"] = (
    df_final["mandatos_cum_norm_w"]
    + df_final["fid_gerais_cum_norm_w"]
    + df_final["tempo_atuacao_percent_norm_w"]
)

for col in ["dim_comprometimento", "dim_carreira"]:
    df_final[f"{col}_norm"] = min_max_normalize(df_final[col])

df_final["ipp"] = (df_final["dim_comprometimento_norm"] + df_final["dim_carreira_norm"]) / 3

In [47]:
df_final.to_excel('./outputs/df_ipp_vf.xlsx')